# 论文 1：复杂动力学第一定律（The First Law of Complexodynamics）
## Scott Aaronson

### 实现：元胞自动机与熵增长

本 Notebook 使用元胞自动机，演示封闭系统中的复杂度和熵如何随时间增长。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import entropy

np.random.seed(42)

## 一维元胞自动机（规则 30——混沌）

In [ ]:
def rule_30(left, center, right):
    """规则 30：生成复杂的混沌图案"""
    pattern = (left << 2) | (center << 1) | right
    rule = 30
    return (rule >> pattern) & 1

def evolve_ca(initial_state, steps, rule_func):
    """让元胞自动机演化指定的步数"""
    size = len(initial_state)
    history = np.zeros((steps, size), dtype=int)
    history[0] = initial_state
    
    for t in range(1, steps):
        for i in range(size):
            left = history[t-1, (i-1) % size]
            center = history[t-1, i]
            right = history[t-1, (i+1) % size]
            history[t, i] = rule_func(left, center, right)
    
    return history

# 简单的初始状态
size = 100
initial = np.zeros(size, dtype=int)
initial[size // 2] = 1  # 只激活中心的一个元胞

# 开始演化
steps = 100
evolution = evolve_ca(initial, steps, rule_30)

plt.figure(figsize=(12, 6))
plt.imshow(evolution, cmap='binary', interpolation='nearest')
plt.title('Rule 30 Cellular Automaton - Complexity Growth from Simple Initial State')
plt.xlabel('Cell Position')
plt.ylabel('Time Step')
plt.colorbar(label='State')
plt.show()

## 使用熵衡量复杂度增长

In [ ]:
def measure_entropy_over_time(history):
    """测量每个时间步的 Shannon 熵"""
    entropies = []
    
    for t in range(len(history)):
        state = history[t]
        # 计算状态的概率分布
        unique, counts = np.unique(state, return_counts=True)
        probs = counts / len(state)
        ent = entropy(probs, base=2)
        entropies.append(ent)
    
    return np.array(entropies)

def measure_spatial_complexity(history):
    """使用相邻元胞的状态转换次数衡量空间图案复杂度"""
    complexities = []
    
    for t in range(len(history)):
        state = history[t]
        # 统计相邻元胞之间的状态转换次数
        transitions = np.sum(np.abs(np.diff(state)))
        complexities.append(transitions)
    
    return np.array(complexities)

entropies = measure_entropy_over_time(evolution)
complexities = measure_spatial_complexity(evolution)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.plot(entropies, linewidth=2)
ax1.set_xlabel('Time Step')
ax1.set_ylabel('Shannon Entropy (bits)')
ax1.set_title('Entropy Growth Over Time')
ax1.grid(True, alpha=0.3)

ax2.plot(complexities, linewidth=2, color='orange')
ax2.set_xlabel('Time Step')
ax2.set_ylabel('Spatial Complexity (transitions)')
ax2.set_title('Spatial Pattern Complexity')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Initial Entropy: {entropies[0]:.4f} bits")
print(f"Final Entropy: {entropies[-1]:.4f} bits")
print(f"Entropy Increase: {entropies[-1] - entropies[0]:.4f} bits")

## 咖啡自动机：不可逆的混合过程

这里展示简单初始状态如何演化成复杂图案——就像奶油混入咖啡一样——而反向过程出现的概率极低。

In [ ]:
def diffusion_2d(grid, steps, diffusion_rate=0.1):
    """简单的二维扩散模拟"""
    history = [grid.copy()]
    
    for _ in range(steps):
        new_grid = grid.copy()
        h, w = grid.shape
        
        for i in range(1, h-1):
            for j in range(1, w-1):
                # 与上下左右四个相邻位置取平均
                neighbors = (
                    grid[i-1, j] + grid[i+1, j] + 
                    grid[i, j-1] + grid[i, j+1]
                ) / 4
                new_grid[i, j] = (
                    (1 - diffusion_rate) * grid[i, j] + 
                    diffusion_rate * neighbors
                )
        
        grid = new_grid
        history.append(grid.copy())
    
    return np.array(history)

# 创建初始状态：咖啡中一小块集中的“奶油”
size = 50
grid = np.zeros((size, size))
grid[20:30, 20:30] = 1.0  # 集中区域

# 模拟混合过程
mixing_history = diffusion_2d(grid, steps=50, diffusion_rate=0.2)

# 可视化混合过程
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
timesteps = [0, 5, 10, 15, 20, 30, 40, 50]

for idx, (ax, t) in enumerate(zip(axes.flat, timesteps)):
    ax.imshow(mixing_history[t], cmap='YlOrBr', vmin=0, vmax=1)
    ax.set_title(f'Time Step {t}')
    ax.axis('off')

plt.suptitle('Irreversible Mixing: The Coffee Automaton', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# 测量混合过程中的熵增长
mixing_entropies = []
for t in range(len(mixing_history)):
    flat = mixing_history[t].flatten()
    # 将连续数值离散化，以便计算熵
    bins = np.histogram(flat, bins=20)[0]
    probs = bins[bins > 0] / bins.sum()
    mixing_entropies.append(entropy(probs, base=2))

plt.figure(figsize=(10, 5))
plt.plot(mixing_entropies, linewidth=2)
plt.xlabel('Time Step')
plt.ylabel('Spatial Entropy (bits)')
plt.title('Entropy Increases During Mixing (Second Law)')
plt.grid(True, alpha=0.3)
plt.show()

print(f"\nKey Insight: Simple concentrated state → Complex mixed state")
print(f"This process is irreversible: you can't unmix coffee!")

## 核心要点

1. **复杂度增长**：简单的初始状态会演化成复杂图案
2. **熵增加**：封闭系统倾向于走向更高熵的状态（热力学第二定律）
3. **不可逆性**：复杂状态几乎不会自发回到简单状态
4. **计算不可逆性**：咖啡自动机展示了这种基本限制

这些内容与深度学习的联系包括：
- 理解信息论
- 理解模型学习到的表示为何具有复杂性
- 理解损失函数和正则化中的熵